In [1]:
!pip install prophet -q

In [2]:
from google.colab import files
uploaded = files.upload()

Saving Results.csv to Results.csv


In [3]:
import pandas as pd
df = pd.read_csv('Results.csv', header = None, names = ['category', 'sales_month', 'total_revenue', 'total_units'])
print(df.shape)
df.head(10)

(192, 4)


,category,sales_month,total_revenue,total_units
0,Apparel,2024-01-01,182984.63,448
1,Apparel,2024-02-01,158336.76,549
2,Apparel,2024-03-01,182704.94,469
3,Apparel,2024-04-01,247061.90,660
4,Apparel,2024-05-01,176944.75,465
5,Apparel,2024-06-01,236502.07,580
6,Apparel,2024-07-01,166295.51,418
7,Apparel,2024-08-01,204136.35,657
8,Apparel,2024-09-01,180543.21,449
9,Apparel,2024-10-01,192596.05,478


In [4]:
grocery_df = df[df['category'] == 'Grocery'] [['sales_month', 'total_revenue']].copy()
grocery_df.columns = ['ds','y']
grocery_df['ds'] = pd.to_datetime(grocery_df['ds'])
grocery_df = grocery_df.sort_values('ds').reset_index(drop = True)
print(grocery_df.shape)
grocery_df.head()

(24, 2)


,ds,y
0,2024-01-01,2122971.50
1,2024-02-01,2069817.34
2,2024-03-01,2285751.03
3,2024-04-01,3515204.33
4,2024-05-01,2327337.72


In [5]:
from prophet import Prophet
model = Prophet(yearly_seasonality = True, weekly_seasonality = False, daily_seasonality = False)
model.fit(grocery_df)

INFO:prophet:n_changepoints greater than number of observations. Using 18.


In [6]:
future = model.make_future_dataframe(periods = 3, freq = 'MS')
forecast = model.predict(future)
forecast[['ds', 'yhat', 'yhat_lower', 'yhat_upper']].tail(6)

,ds,yhat,yhat_lower,yhat_upper
21,2025-10-01,2.768729e+06,2.768729e+06,2.768729e+06
22,2025-11-01,2.951900e+06,2.951900e+06,2.951900e+06
23,2025-12-01,3.533283e+06,3.533283e+06,3.533283e+06
24,2026-01-01,3.111239e+06,3.106984e+06,3.115592e+06
25,2026-02-01,2.726929e+06,2.713367e+06,2.741865e+06
26,2026-03-01,3.311674e+06,3.285178e+06,3.341904e+06


In [7]:
train_df= grocery_df[:-3]
test_df = grocery_df[-3:]
print("Train Size: ", train_df.shape)
print("Test Size: ", test_df.shape)

backtest_model = Prophet(yearly_seasonality = True, weekly_seasonality = False, daily_seasonality = False)
backtest_model.fit(train_df)

backtest_future = backtest_model.make_future_dataframe(periods = 3, freq = 'MS')
backtest_forecast = backtest_model.predict(backtest_future)

comparison = backtest_forecast[['ds', 'yhat']].tail(3).reset_index(drop = True)
comparison['actual'] = test_df['y'].values
comparison['error_pct'] = (comparison['yhat'] - comparison['actual']) / comparison['actual'] * 100

comparison

INFO:prophet:n_changepoints greater than number of observations. Using 15.


Train Size:  (21, 2)
Test Size:  (3, 2)


,ds,yhat,actual,error_pct
0,2025-10-01,4.396785e+06,2768728.82,58.801578
1,2025-11-01,2.697066e+06,2951900.34,-8.632902
2,2025-12-01,3.795584e+06,3533282.60,7.423740


In [12]:
import logging
logging.getLogger('prophet').setLevel(logging.ERROR)
logging.getLogger('cmdstanpy').setLevel(logging.ERROR)

categories = df['category'].unique()
results = []

for cat in categories:
    cat_df = df[df['category'] == cat][['sales_month', 'total_revenue']].copy()
    cat_df.columns = ['ds', 'y']
    cat_df['ds'] = pd.to_datetime(cat_df['ds'])
    cat_df = cat_df.sort_values('ds').reset_index(drop=True)

    full_model = Prophet(yearly_seasonality=True, weekly_seasonality=False, daily_seasonality=False)
    full_model.fit(cat_df)
    future = full_model.make_future_dataframe(periods=3, freq='MS')
    forecast = full_model.predict(future)
    next_3_months = forecast[['ds', 'yhat']].tail(3)

    train_df = cat_df[:-3]
    test_df = cat_df[-3:]
    bt_model = Prophet(yearly_seasonality=True, weekly_seasonality=False, daily_seasonality=False)
    bt_model.fit(train_df)
    bt_future = bt_model.make_future_dataframe(periods=3, freq='MS')
    bt_forecast = bt_model.predict(bt_future)
    bt_predicted = bt_forecast['yhat'].tail(3).values
    bt_actual = test_df['y'].values
    avg_error_pct = ((bt_predicted - bt_actual) / bt_actual * 100).mean()

    for i, row in enumerate(next_3_months.itertuples()):
        results.append({
            'category': cat,
            'forecast_month': row.ds.strftime('%Y-%m'),
            'forecast_revenue': round(row.yhat, 0),
            'backtest_avg_error_pct': round(avg_error_pct, 1)
        })

# এই দুই লাইন এখন loop-এর বাইরে (এক ধাপ কম indent) — শুধু loop সম্পূর্ণ শেষ হলে একবার চলবে
summary_df = pd.DataFrame(results)
summary_df

,category,forecast_month,forecast_revenue,backtest_avg_error_pct
0,Apparel,2026-01,218876.0,25.4
1,Apparel,2026-02,199823.0,25.4
2,Apparel,2026-03,218266.0,25.4
3,Beverages,2026-01,868829.0,18.5
4,Beverages,2026-02,805859.0,18.5
5,Beverages,2026-03,918011.0,18.5
6,Electronics Accessories,2026-01,220976.0,6.5
7,Electronics Accessories,2026-02,192255.0,6.5
8,Electronics Accessories,2026-03,241279.0,6.5
9,Footwear,2026-01,151176.0,24.9
